In [0]:
from pyspark.sql import functions as F

# Load all 560 critical alerts saved from Notebook 2
alerts_df = spark.table("workspace.default.critical_fraud_alerts")

print(f"Total Critical Alerts (>90% Risk) Loaded: {alerts_df.count()}")

# Display ALL 560 alerts in Databricks interactive table
display(alerts_df)


Total Critical Alerts (>90% Risk) Loaded: 560


step,type,nameOrig,nameDest,amount,oldbalanceOrg,newbalanceOrig,balance_drop,is_account_emptied,fraud_prob,isFraud
13,TRANSFER,C1864459372,C537531657,1161495.82,1161495.82,0.0,1161495.82,1,0.9766,1
26,TRANSFER,C414159853,C1087566727,1027152.36,1027152.36,0.0,1027152.36,1,0.9766,1
34,TRANSFER,C638438531,C2146216948,1016540.35,1016540.35,0.0,1016540.35,1,0.9766,1
51,TRANSFER,C1948396559,C538567441,1170282.92,1170282.92,0.0,1170282.92,1,0.9766,1
53,TRANSFER,C101455775,C1834538811,931684.75,931684.75,0.0,931684.75,1,0.9766,1
64,TRANSFER,C1837436553,C611839321,841052.98,841052.98,0.0,841052.98,1,0.9766,1
70,TRANSFER,C282915164,C672400159,1036789.41,1036789.41,0.0,1036789.41,1,0.9766,1
75,TRANSFER,C756454870,C1095120189,810039.19,810039.19,0.0,810039.19,1,0.9766,1
90,TRANSFER,C787624212,C707993034,1182557.97,1182557.97,0.0,1182557.97,1,0.9766,1
91,TRANSFER,C1152872254,C1484049545,921668.95,921668.95,0.0,921668.95,1,0.9766,1


In [0]:
%pip install -q google-genai


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


In [0]:
import json
from google import genai

GEMINI_API_KEY = "YOUR_API_KEY_HERE"
client = genai.Client(api_key=GEMINI_API_KEY)

top_alerts = alerts_df.limit(5).toPandas().to_dict(orient="records")

prompt = f"""
You are a senior bank fraud compliance analyst. Review these 5 flagged high-risk transactions and write a concise 2-sentence Suspicious Activity Report (SAR) for each one explaining why it is high risk and what action to take.

Flagged Transactions JSON:
{json.dumps(top_alerts, indent=2)}
"""

# Automatically find all text-generation models enabled for your API key
available_models = [
    m.name for m in client.models.list() 
    if "generateContent" in (m.supported_actions or []) and ("flash" in m.name or "gemma" in m.name)
]
print("Models available on your key:", available_models)

for model_name in available_models:
    try:
        print(f"\nTrying {model_name}...")
        response = client.models.generate_content(model=model_name, contents=prompt)
        print("\n=== AUTOMATED GENAI FRAUD INVESTIGATION REPORTS ===\n")
        print(response.text)
        break
    except Exception as e:
        print(f"{model_name} busy ({e}), trying next model...")


Models available on your key: ['models/gemini-2.5-flash', 'models/gemma-4-26b-a4b-it', 'models/gemma-4-31b-it', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash']

Trying models/gemini-2.5-flash...
models/gemini-2.5-flash busy (404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}), trying next model...

Trying models/gemma-4-26b-a4b-it...

=== AUTOMATED GENAI FRAUD INVESTIGATION REPORTS ===

**Transaction 1 (C18644593